In [29]:
xml_path = "../../Editions/Guerin_tome1-tome12.xml"
csv_path = "Guerin-csv.csv"



In [39]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
tei_to_csv.py

Extrait les <text> d'une édition TEI (teiCorpus/TEI/text/group/text)
vers un CSV, avec conversion du balisage mixte TEI -> HTML pour les
champs 'abstract' (argument) et 'edition' (transcription).

Usage:
    python3 tei_to_csv.py input.xml output.csv
"""

import csv
import re
import sys
from lxml import etree

TEI_URI = "http://www.tei-c.org/ns/1.0"
XML_URI = "http://www.w3.org/XML/1998/namespace"
NS = {"tei": TEI_URI}


def qxml(attr):
    return f"{{{XML_URI}}}{attr}"


def local(tag):
    return etree.QName(tag).localname


def strip_leading_zeros(s):
    s = s or ""
    stripped = s.lstrip("0")
    return stripped if stripped else "0"


def escape_text(text):
    if text is None:
        return ""
    return (
        text.replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
    )


# ---------------------------------------------------------------------
# Conversion récursive des éléments TEI -> HTML
# ---------------------------------------------------------------------

# Mapping des valeurs de @rend sur <hi> vers des balises HTML sémantiques
HI_REND_MAP = {
    "sup": "sup",
    "sub": "sub",
    "i": "em",
    "italic": "em",
    "b": "strong",
    "bold": "strong",
}

def get_head_text(elem):
    if elem is None:
        return ""

    parts = []

    if elem.text:
        parts.append(elem.text)

    for child in elem:
        if isinstance(child.tag, str):
            # Pour emph, hi, etc. : on colle le contenu au texte précédent
            parts.append(get_head_text(child))

        if child.tail:
            parts.append(child.tail)

    text = "".join(parts)

    # Normalisation des espaces XML
    text = re.sub(r"\s+", " ", text).strip()

    # Dans les titres, "DLVI <emph>bis</emph>" doit devenir "DLVIbis"
    text = re.sub(r"\s+(?=(bis|ter|quater)\b)", "", text, flags=re.IGNORECASE)

    return text

def normalize_text_fragment(text):
    """
    Nettoie un fragment de texte XML sans supprimer
    les espaces nécessaires entre les éléments.

    Équivalent pratique de normalize-space(), adapté
    au texte mixte TEI.
    """
    if not text:
        return ""

    # Tous les espaces XML (espace, tabulation, retour ligne)
    # deviennent un espace simple.
    text = re.sub(r"\s+", " ", text)

    return text

def serialize_children(elem, footnotes, note_prefix, note_counter):
    parts = []

    if elem.text:
        parts.append(normalize_text_fragment(elem.text))

    for child in elem:

        # Ignore commentaires XML
        if isinstance(child.tag, str):
            parts.append(
                serialize_element(
                    child,
                    footnotes,
                    note_prefix,
                    note_counter
                )
            )

        if child.tail:
            parts.append(normalize_text_fragment(child.tail))

    return "".join(parts)

def serialize_element(elem, footnotes, note_prefix, note_counter):
    # Ignore les commentaires XML et autres nœuds non-éléments
    # (ex. <?xml-stylesheet ...?>)
    if not isinstance(elem.tag, str):
        return ""

    tag = local(elem.tag)
    inner = serialize_children(elem, footnotes, note_prefix, note_counter)

    if tag == "p":
        return f"<p>{inner}</p>"

    if tag == "note":
        note_counter[0] += 1
        n = note_counter[0]
        note_id = f"{note_prefix}-{n}"
        footnotes.append((note_id, inner))
        return (
            f'<sup class="tei-note-ref">'
            f'<a id="ref-{note_id}" href="#note-{note_id}">{n}</a></sup>'
        )

    if tag == "quote":
        return f'<q class="tei-quote">{inner}</q>'

    if tag == "hi":
        rend = elem.get("rend", "")
        if rend in HI_REND_MAP:
            htag = HI_REND_MAP[rend]
            return f"<{htag}>{inner}</{htag}>"
        cls = f' class="rend-{rend}"' if rend else ""
        return f"<span{cls}>{inner}</span>"

    if tag == "num":
        return f'<span class="tei-num">{inner}</span>'

    if tag == "pb":
        return ""

    if tag == "milestone":
        return ""

    if tag == "lb":
        return "<br/>"

    if tag == "foreign":
        lang = elem.get(qxml("lang"), "")
        attrs = f' lang="{lang}"' if lang else ""
        return f'<span class="tei-foreign"{attrs}>{inner}</span>'

    # Filet de sécurité : aucun élément TEI n'est perdu
    return f'<span class="tei-{tag}">{inner}</span>'



def extract_field_html(container, note_prefix):
    """
    Convertit un conteneur TEI en un seul bloc HTML :
    texte principal + notes de bas de page à la suite.
    """
    if container is None:
        return ""

    footnotes = []
    counter = [0]
    parts = []

    for child in container:
        if isinstance(child.tag, str):
            parts.append(
                serialize_element(
                    child,
                    footnotes,
                    note_prefix,
                    counter
                )
            )

    main_html = "".join(parts).strip()

    # Ajout des notes à la suite du texte principal
    if footnotes:
        items = "".join(
            f'<li id="note-{nid}">{html} '
            f'<a href="#ref-{nid}" class="tei-note-backref">&#8617;</a></li>'
            for nid, html in footnotes
        )

        footnotes_html = (
            f'<ol class="tei-footnotes">{items}</ol>'
        )

        main_html += footnotes_html

    return main_html

# ---------------------------------------------------------------------
# Extraction principale
# ---------------------------------------------------------------------

FIELDNAMES = [
    "biblId",
    "nROM",
    "head",
    "xml_id",
    "tome",
    "n",
    "registre_idno",
    "locus",
    "act_number",
    "auth_type",
    "act_type",
    "date",
    "date_non_standard",
    "abstract",
    "edition",
    "lang",

]

def extract_tradition_info(t):
    """Extrait, pour chaque temoin manuscrit (listWit/witness contenant
    un msDesc), l'idno du manuscrit, le locus et le numero d'acte.
    Plusieurs temoins -> valeurs jointes par '|'."""
    registre_idnos = []
    loci = []
    act_numbers = []

    witnesses = t.findall(
        './/tei:front/tei:div[@type="tradition"]/tei:listWit/tei:witness',
        NS,
    )
    for w in witnesses:
        msdesc = w.find("./tei:msDesc", NS)
        if msdesc is None:
            continue  # temoin sans manuscrit (ex. temoin d'edition imprimee)

        idno_ms_el = msdesc.find("./tei:msIdentifier/tei:idno", NS)
        registre_idnos.append(
            (idno_ms_el.text or "").strip() if idno_ms_el is not None else ""
        )

        locus_el = w.find("./tei:locus", NS)
        loci.append(
            (locus_el.text or "").strip() if locus_el is not None else ""
        )

        # idno direct de <witness> (numero d'acte), distinct de celui
        # trouve dans msDesc/msIdentifier/idno (identifiant du registre)
        act_idno_el = w.find("./tei:idno", NS)
        act_numbers.append(
            (act_idno_el.text or "").strip() if act_idno_el is not None else ""
        )

    return (
        "|".join(registre_idnos),
        "|".join(loci),
        "|".join(act_numbers),
    )




def process_text_element(t):
    xml_id = t.get(qxml("id"), "")
    n_rom = t.get("n", "")

    head_el = t.find(".//tei:front/tei:head", NS)
    head = get_head_text(head_el)


    # @n = tome4_0556bis
    m = re.match(r"tome(\d+)_(\d+)(.*)$", xml_id or "")

    tome = m.group(1) if m else ""
    n_val = (
        strip_leading_zeros(m.group(2)) + m.group(3)
        if m
        else ""
    )

    auth_terms = t.findall('.//tei:index/tei:term[@type="auth_type"]', NS)
    auth_type = ";".join(term.get("key", "") for term in auth_terms)

    act_terms = t.findall('.//tei:index/tei:term[@type="act_type"]', NS)
    act_type = ";".join(term.get("key", "") for term in act_terms)

    date_el = t.find(".//tei:docDate/tei:date", NS)
    date_when = date_el.get("when", "") if date_el is not None else ""
    date_ns = (date_el.text or "").strip() if date_el is not None else ""

    argument_el = t.find(".//tei:front/tei:argument", NS)
   
    abstract_html = extract_field_html(
        argument_el, f"{xml_id}-abs"
    )
    
    transcription_el = t.find(
        './/tei:body/tei:div[@type="transcription"]', NS
        )
    
    edition_html = extract_field_html(
       transcription_el, f"{xml_id}-ed"
       )
    
    lang = (
        transcription_el.get(qxml("lang"), "")
        if transcription_el is not None
        else ""
    )

    registre_idno, locus, act_number = extract_tradition_info(t)

    return {
        "biblId": "Guérin, Poitou",
        "nROM": n_rom,
        "head": head,
        "xml_id": xml_id,
        "tome": tome,
        "n": n_val,
        "registre_idno": registre_idno,
        "locus": locus,
        "act_number": act_number,
        "auth_type": auth_type,
        "act_type": act_type,
        "date": date_when,
        "date_non_standard": date_ns,
        "lang": lang,
        "abstract": abstract_html,
        "edition": edition_html,
        
        }



In [40]:

def process_tei(xml_path, csv_path):
    parser = etree.XMLParser(recover=True, huge_tree=True)
    tree = etree.parse(xml_path, parser)
    root = tree.getroot()

    # Cherche les <text> a n'importe quel niveau sous group (robuste
    # aux variations de structure teiCorpus/TEI/text/group/text)
    texts = root.findall(".//tei:group/tei:text", NS)
    if not texts:
        # fallback si un seul TEI/text sans group, ou structure differente
        texts = root.findall(".//tei:text", NS)
        # exclut le <text> racine qui contient <group> lui-meme, si present
        texts = [t for t in texts if t.find("./tei:group", NS) is None]

    rows = [process_text_element(t) for t in texts]

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        writer.writerows(rows)

    return len(rows)

'''
if __name__ == "__main__":
    if len(sys.argv) != 3:
        print("Usage: python3 tei_to_csv.py input.xml output.csv")
        sys.exit(1)
    count = process_tei(sys.argv[1], sys.argv[2])
    print(f"{count} enregistrements exportes vers {sys.argv[2]}")
'''
process_tei(xml_path, csv_path)

1746